In [47]:
# Installing libraries
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [48]:
# Load Data and Read data
df=pd.read_csv('mpg.csv')
# Understand Data Shape
print(df.shape)
df.head()

(398, 9)


,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


In [49]:
# #############################################################
# --- Data Cleaning and Preprocessing ---
# #############################################################

# 1. Handle missing 'horsepower' values
# Convert horsepower to numeric (coercing '?' or invalid strings to NaN)
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')

# Fill missing horsepower with column mean
df['horsepower'] = df['horsepower'].fillna(df['horsepower'].mean())

# Display summaries for mpg and horsepower
print("\n--- Summary: MPG ---")
print(df['mpg'].describe())

print("\n--- Summary: Horsepower ---")
print(df['horsepower'].describe())

# 2. Map origin codes (1=USA, 2=Europe, 3=Japan)
origin_map = {1: 'USA', 2: 'Europe', 3: 'Japan'}
df['origin'] = df['origin'].map(origin_map)

# Convert origin to category type
#df['origin'] = df['origin'].astype('category')

# 3. Convert cylinders to categorical string labels
df['cylinders'] = df['cylinders'].astype(str).astype('category')

print(f"\n--- Data Loading and Cleaning Complete ({len(df)} records loaded) ---\n")



--- Summary: MPG ---
count    398.000000
mean      23.514573
std        7.815984
min        9.000000
25%       17.500000
50%       23.000000
75%       29.000000
max       46.600000
Name: mpg, dtype: float64

--- Summary: Horsepower ---
count    398.000000
mean     104.469388
std       38.199187
min       46.000000
25%       76.000000
50%       95.000000
75%      125.000000
max      230.000000
Name: horsepower, dtype: float64

--- Data Loading and Cleaning Complete (398 records loaded) ---



In [50]:
# #############################################################
# I. DESCRIPTIVE ANALYSIS
# #############################################################

print("=============================================================")
print("I. DESCRIPTIVE ANALYSIS: Summarizing the Data")
print("=============================================================")

# 1. Overall Numerical Summary (Formatted)
# Provides mean, standard deviation, min/max, and quartiles.
print("\n1. Summary Statistics for Key Numerical Variables:")
num_vars = ['mpg', 'weight', 'horsepower', 'displacement']

# Calculate summary, transpose, and round to 2 decimal places
summary = df[num_vars].describe().T.round(2)

# FIX: Replaced deprecated applymap with map for element-wise formatting
summary = summary.map(lambda x: f"{x:.2f}")

print(summary)

# 2. Grouped Descriptive Statistics
# Analyze the primary dependent variable (MPG) broken down by car 'origin'.
print("\n2. Mean, Standard Deviation, and Count of MPG Grouped by Origin:")
# FIX: Grouping works correctly now that 'origin' is a standard string/object type
print(df.groupby('origin')['mpg'].agg(['min', 'max', 'median', 'mean', 'std', 'count']).round(2))

# 3. Correlation Matrix
# Shows the linear relationship (Pearson's r) between key variables.
print("\n3. Correlation Matrix (Pearson's r) for Fuel Economy Predictors:")
numerical_cols = ['mpg', 'weight', 'horsepower', 'displacement', 'acceleration']
# Note: df is now clean due to the imputation above, ensuring correlation calculation works.
print(df[numerical_cols].corr().round(3))


I. DESCRIPTIVE ANALYSIS: Summarizing the Data

1. Summary Statistics for Key Numerical Variables:
               count     mean     std      min      25%      50%      75%  \
mpg           398.00    23.51    7.82     9.00    17.50    23.00    29.00   
weight        398.00  2970.42  846.84  1613.00  2223.75  2803.50  3608.00   
horsepower    398.00   104.47   38.20    46.00    76.00    95.00   125.00   
displacement  398.00   193.43  104.27    68.00   104.25   148.50   262.00   

                  max  
mpg             46.60  
weight        5140.00  
horsepower     230.00  
displacement   455.00  

2. Mean, Standard Deviation, and Count of MPG Grouped by Origin:
         min   max  median   mean   std  count
origin                                        
Europe  16.2  44.3    26.5  27.89  6.72     70
Japan   18.0  46.6    31.6  30.45  6.09     79
USA      9.0  39.0    18.5  20.08  6.40    249

3. Correlation Matrix (Pearson's r) for Fuel Economy Predictors:
                mpg  weight  

In [51]:

# #############################################################
# II. INFERENTIAL ANALYSIS
# #############################################################

print("\n\n=============================================================")
print("II. INFERENTIAL ANALYSIS: Testing Relationships")
print("=============================================================")
alpha = 0.05 # Significance level

# 1. T-Test (Comparing Two Group Means)
# Goal: Is there a significant difference in fuel efficiency (MPG) between USA and European cars?
usa_mpg = df[df['origin'] == 'USA']['mpg']
europe_mpg = df[df['origin'] == 'Europe']['mpg']

# Use Welch's T-Test (t-test_ind with equal_var=False) as a robust choice.
t_stat, p_value_ttest = stats.ttest_ind(usa_mpg, europe_mpg, equal_var=False)

print("\n1. Independent Samples T-Test (USA MPG vs Europe MPG):")
print(f"   H₀: $\\mu_{{USA}} = \\mu_{{Europe}}$ (No difference in mean MPG)")
print(f"   T-Statistic: {t_stat:.4f}")
print(f"   P-Value: {p_value_ttest:.5f}")

if p_value_ttest < alpha:
    print(f"   Conclusion: **Reject H₀**. The difference in mean MPG is statistically significant (P < {alpha}).")
    direction = "lower" if usa_mpg.mean() < europe_mpg.mean() else "higher"
    print(f"   Observation: USA cars have a significantly {direction} mean MPG compared to European cars.")
else:
    print(f"   Conclusion: **Fail to Reject H₀**. No statistically significant difference.")





II. INFERENTIAL ANALYSIS: Testing Relationships

1. Independent Samples T-Test (USA MPG vs Europe MPG):
   H₀: $\mu_{USA} = \mu_{Europe}$ (No difference in mean MPG)
   T-Statistic: -8.6726
   P-Value: 0.00000
   Conclusion: **Reject H₀**. The difference in mean MPG is statistically significant (P < 0.05).
   Observation: USA cars have a significantly lower mean MPG compared to European cars.


In [54]:
# 2. ANOVA (Comparing Multiple Group Means)
# Goal: Does the number of cylinders (4, 6, 8) significantly affect MPG?
# The formula specifies that MPG is the dependent variable explained by the categorical variable C(cylinders).
alpha = 0.05  # Significance level

print("\n2. One-Way ANOVA: MPG by Cylinder Category")

# OLS model with categorical predictors
model_anova = smf.ols("mpg ~ C(cylinders)", data=df).fit()

# Correct ANOVA function (statsmodels.stats.api)
anova_table = sm.stats.anova_lm(model_anova, typ=2)

# Display hypothesis and results
print("   H₀: μ₍4-cyl₎ = μ₍6-cyl₎ = μ₍8-cyl₎ (All group means are equal)")
print("\n   ANOVA Summary Table:")
print(anova_table)

# Extract the P-value
anova_p = anova_table.loc['C(cylinders)', 'PR(>F)']

# Interpretation
if anova_p < alpha:
    print(f"\n   Conclusion: **Reject H₀**. Cylinders significantly affect MPG (p = {anova_p:.6f}).")
else:
    print(f"\n   Conclusion: **Fail to Reject H₀**. No statistically significant effect (p = {anova_p:.6f}).")


2. One-Way ANOVA: MPG by Cylinder Category
   H₀: μ₍4-cyl₎ = μ₍6-cyl₎ = μ₍8-cyl₎ (All group means are equal)

   ANOVA Summary Table:
                    sum_sq     df           F        PR(>F)
C(cylinders)  15454.761883    4.0  172.591785  3.679939e-85
Residual       8797.813594  393.0         NaN           NaN

   Conclusion: **Reject H₀**. Cylinders significantly affect MPG (p = 0.000000).


In [55]:
# #############################################################
# III. PREDICTIVE ANALYSIS
# #############################################################

print("\n\n=============================================================")
print("III. PREDICTIVE ANALYSIS: Forecasting with Regression")
print("=============================================================")

# 1. Fit the Model: MPG is predicted by Weight, Horsepower, and Origin
# The C(origin) term tells statsmodels to treat 'origin' as a categorical factor,
# creating dummy variables automatically.
model_reg = smf.ols("mpg ~ weight + horsepower + C(origin)", data=df).fit()

print("\n1. Multiple Linear Regression (MPG ~ Weight + HP + Origin)")
print(f"   Overall Model Fit (R-squared): {model_reg.rsquared:.4f}")
print(f"   Interpretation: The predictors explain approximately {model_reg.rsquared*100:.2f}% of the variance in MPG.")

# 2. Interpretation of Predictor Coefficients
print("\n2. Interpretation of Key Predictor Coefficients:")
reg_summary = model_reg.summary2().tables[1] # Extract the coefficients table

# Weight coefficient
weight_coef = reg_summary.loc['weight', 'Coef.']
weight_p = reg_summary.loc['weight', 'P>|t|']
print(f"   Weight Coefficient: {weight_coef:.4f} (P-Value: {weight_p:.4f})")
print(f"   -> Meaning: Holding all else constant, every 1-unit increase in **Weight** reduces MPG by {abs(weight_coef):.4f}.")

# 3. Prediction for a New Observation
# Scenario: A new car built in Japan with specific specs.
new_car = pd.DataFrame({
    'weight': [2000],
    'horsepower': [90],
    'origin': ['Japan']
})

# Get prediction results
predictions = model_reg.get_prediction(new_car)

# Extract predicted mean and 95% prediction interval (PI)
predicted_mpg = predictions.summary_frame(alpha=0.05)['mean'][0]
pi_lower = predictions.summary_frame(alpha=0.05)['obs_ci_lower'][0]
pi_upper = predictions.summary_frame(alpha=0.05)['obs_ci_upper'][0]

print("\n3. Prediction for a Hypothetical Car:")
print(f"   Specs: Weight=2000 lbs, Horsepower=90 HP, Origin=Japan")
print(f"   Predicted MPG (Point Estimate): {predicted_mpg:.2f} MPG")
print(f"   95% Prediction Interval (PI):   [{pi_lower:.2f}, {pi_upper:.2f}] MPG")
print("\n   Interpretation: We are 95% confident that the actual MPG for a car with these specifications will fall within this interval.")



III. PREDICTIVE ANALYSIS: Forecasting with Regression

1. Multiple Linear Regression (MPG ~ Weight + HP + Origin)
   Overall Model Fit (R-squared): 0.7162
   Interpretation: The predictors explain approximately 71.62% of the variance in MPG.

2. Interpretation of Key Predictor Coefficients:
   Weight Coefficient: -0.0050 (P-Value: 0.0000)
   -> Meaning: Holding all else constant, every 1-unit increase in **Weight** reduces MPG by 0.0050.

3. Prediction for a Hypothetical Car:
   Specs: Weight=2000 lbs, Horsepower=90 HP, Origin=Japan
   Predicted MPG (Point Estimate): 31.07 MPG
   95% Prediction Interval (PI):   [22.78, 39.36] MPG

   Interpretation: We are 95% confident that the actual MPG for a car with these specifications will fall within this interval.
